# Course Project: Navigation

课程大作业要求 Unitree G1 在 5x5 程序化布局中直接输出 29 维关节动作。每个
tile 为 10 m x 10 m，内部布局为 50 m x 50 m；外围四周各有 10 m 平地边界，
所以完整地形为 70 m x 70 m。Actor 只观察 body-frame 下一 waypoint，不接收
route index、全局路线或 progress。使用 Conda `summer` kernel，并在该环境中
用 pip 安装 `mjlab==1.5.0`。提交文件夹只包含 `policy.pt`、`model.py`、`student.py`。

满分 100：提交与有限动作 20；
$30\,\text{route progress}+10\,\text{route success}$；AMP 10；depth 10；
平滑度 20。正式评测使用 10 个不重复的随机起点。Motion 的来源、许可和哈希
与 Experiment 07 相同，见
`assets/motions/manifest.json` 与 `NOTICE.txt`。

In [ ]:
%load_ext autoreload
%autoreload 2

import importlib.util
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from IPython.display import Video, display
from tqdm.auto import tqdm


def find_exp_root(name: str) -> Path:
  for candidate in (Path.cwd(), *Path.cwd().parents):
    if candidate.name == name:
      return candidate
    nested = candidate / name
    if nested.is_dir():
      return nested
  raise FileNotFoundError(name)


def load_student(path: Path):
  spec = importlib.util.spec_from_file_location("active_student", path)
  if spec is None or spec.loader is None:
    raise ImportError(path)
  module = importlib.util.module_from_spec(spec)
  spec.loader.exec_module(module)
  return module


assert Path(sys.prefix).name == "summer", "请切换到 Conda summer kernel"
assert version("mjlab") == "1.5.0"

PROJECT_ROOT = find_exp_root("course_project")
COURSE_ROOT = PROJECT_ROOT.parent
STUDENT_FILE = PROJECT_ROOT / "student.py"
MODE = "height"  # 可改为 "depth"
sys.path.insert(0, str(PROJECT_ROOT))

from src import workflow  # noqa: E402
from src.mjlab_tasks.env_cfgs import (  # noqa: E402
  course_g1_navigation_env_cfg,
)
from src.terrain import (  # noqa: E402
  generate_navigation_scene,
  sample_evaluation_seeds,
)

scene = generate_navigation_scene(seed=23)
cfg = course_g1_navigation_env_cfg(MODE, student_path=STUDENT_FILE, scene_seed=23)
assert (scene.rows, scene.cols, scene.tile_size) == (5, 5, 10.0)
assert scene.route_length <= 100.0
assert cfg.auto_reset is False and set(cfg.actions) == {"joint_pos"}
evaluation_seeds = sample_evaluation_seeds(101, count=10)
evaluation_starts = [generate_navigation_scene(s).route[0] for s in evaluation_seeds]
assert len(set(evaluation_starts)) == 10
print("环境初始化完成：tile=10 m，layout=50 m，total=70 m")
print("29 维动作，83 维 AMP state")
print("10 个评测起点：", evaluation_starts)


## AMP state 与 depth

`build_amp_state()` 与 Experiment 07 公式相同，但必须在课程大作业中独立实现。
Depth 裁剪到 $[0.1,5.0]\,\text{m}$ 并映射到 $[0,1]$。Depth actor 接收
`[B,1,60,80]`，height baseline 接收 height scan。

In [ ]:
s = load_student(STUDENT_FILE)
parts = (
  torch.zeros(2, 29), torch.ones(2, 29), torch.ones(2, 1),
  torch.zeros(2, 3), torch.zeros(2, 3), torch.zeros(2, 3),
  torch.zeros(2, 5, 3),
)
state = s.build_amp_state(*parts)
assert state.shape == (2, 83) and torch.isfinite(state).all()
print("代码检查通过；build_amp_state 对应 AMP 10 分")
depth = s.normalize_depth(torch.tensor([[[[0.0, 0.1, 5.0, 8.0]]]]))
torch.testing.assert_close(depth, torch.tensor([[[[0.0, 0.0, 1.0, 1.0]]]]))
print("代码检查通过；normalize_depth 对应可选 depth 10 分")


## Waypoint、任务奖励与平滑度

将世界坐标 waypoint 用 base yaw 的逆旋转变换到 body frame。任务奖励组合 route
progress、到达 waypoint 和 route success：
$r_t=4\,\text{progress}_t+0.5\,\text{waypoint reached}_t
+5\,\text{route success}_t$。平滑度为
$\text{mean}(|a_t-2a_{t-1}+a_{t-2}|)$。

In [ ]:
s = load_student(STUDENT_FILE)
half = 2 ** -0.5
body = s.waypoint_in_body_frame(
  torch.zeros(1, 3), torch.tensor([[half, 0.0, 0.0, half]]),
  torch.tensor([[1.0, 0.0, 0.0]]),
)
torch.testing.assert_close(body, torch.tensor([[0.0, -1.0]]), atol=1e-5, rtol=1e-5)
print("代码检查通过；waypoint_in_body_frame 是 actor observation 合同")
reward = s.navigation_reward(
  torch.tensor([0.1]), torch.tensor([True]), torch.tensor([False])
)
assert reward.shape == (1,) and torch.isfinite(reward).all()
print("代码检查通过；navigation_reward 对应任务 40 分")
actions = torch.ones(3, 29)
torch.testing.assert_close(
  s.smoothness_penalty(actions, actions, actions), torch.zeros(3)
)
print("代码检查通过；smoothness_penalty 对应平滑度 20 分")


## 场景与 smoke

精简 terrain 工具只保留 portal、route graph、最小 WFC、pile、platform gap、
pyramid stairs 和 mjlab SubTerrain adapter。5x5 grid 的每个 tile 边长为 10 m，
外围平地边界宽 10 m，总体尺寸为 70 m x 70 m；route 起点由 seed 随机产生。
先可视化 tile 与路线，再运行 32-env 物理 smoke；图中路线只用于理解场景，
不会作为 actor privileged input。

In [ ]:
%%time
display(workflow.plot_scene(scene))
smoke_result = workflow.smoke(
  MODE, num_envs=32, steps=16, device="cuda:0",
  student_file=STUDENT_FILE, force_termination=True,
)
display(smoke_result)


## Navigation AMP-PPO 训练

Height 默认 2048 env，depth 先使用 32 env。训练直接优化 29 维关节策略；可选的
层级速度策略只是扩展接口，不参与必做评分。

In [ ]:
%%time
TRAIN_ENVS = 2048 if MODE == "height" else 32
RUN_DIR = workflow.train(
  MODE, num_envs=TRAIN_ENVS, iterations=1000, steps_per_env=24,
  device="cuda:0", seed=23, student_file=STUDENT_FILE,
)
print(RUN_DIR)


In [ ]:
%%time
CHECKPOINT = workflow.latest_checkpoint()
metrics = workflow.evaluate(
  CHECKPOINT, MODE, num_envs=1, steps=5000, evaluations=10,
  device="cuda:0", seed=101, student_file=STUDENT_FILE,
)
display(metrics)
plot_metrics = {
  key: metrics[key] for key in ("route_progress", "route_success", "smoothness")
}
figure, axis = plt.subplots(figsize=(7, 3.5))
axis.bar(
  plot_metrics,
  plot_metrics.values(),
  color=["#2878b5", "#2a9d8f", "#f4a261"],
)
axis.set_title("Course Project: 10 random-start evaluations")
axis.grid(axis="y", alpha=0.2)
display(figure)
for checkpoint in tqdm([CHECKPOINT], desc="录制 150 帧视频"):
  video_path = workflow.record_video(
    checkpoint, MODE, frames=150, device="cuda:0",
    seed=101, student_file=STUDENT_FILE,
  )
display(Video(str(video_path), embed=True))


In [ ]:
%%time
submission = workflow.prepare_submission(
  CHECKPOINT, MODE, device="cuda:0", student_file=STUDENT_FILE
)
print("submission:", submission)
subprocess.run(
  [sys.executable, str(COURSE_ROOT / "grading_toolkit" / "grade.py"),
   str(submission), "--task", "course_project", "--device", "cuda:0"],
  cwd=COURSE_ROOT,
  check=True,
)
